# 대분류(6종) 카테고리 + 색상 고도화 검증 노트북

**스파이크/검증용 코드입니다. 프로덕션 코드가 아닙니다.**

`product_info_check.ipynb`(소분류 30종 + 카테고리 프롬프트 v1~v4 검증)와 별개 노트북이다 —
이번엔 **대분류(OUTER/TOP/BOTTOM/SHOES/BAG/ACCESSORY) 6종만** 분류했을 때 CLIP과 VLM이
각각 어떤지 비교하고, 색상은 지금까지 나온 개선안(프롬프트 v2/v3 + 화이트밸런스 전처리 +
팔레트 스와치 참조)을 한 자리에서 재실측한다.

## 2026-08-03 추가 — 패션 도메인 특화 체크포인트 스윕(§2-2, §6-1)

여기까지의 실측에서 **CLIP 쪽 개선 시도는 프롬프트·앙상블·스와치가 전부 회귀**했고
(`REPORT.md` §11/§12/§16), 유일하게 재현된 개선은 화이트밸런스 전처리(§13/§17)뿐이었다.
카테고리 대분류 직접 질의는 0.35, 색상은 0.41에서 막혀 있다. 남은 축은 **체크포인트 자체를
바꾸는 것**이라, 지금까지 쓰던 범용 체크포인트(OpenAI CLIP / Google SigLIP) 대신 **패션 상품
이미지로 파인튜닝된 체크포인트 4종**을 같은 정답셋·같은 프롬프트로 돌려 비교한다.

| 키 | 체크포인트 | 고른 이유 |
| --- | --- | --- |
| `fashion_clip` | `patrickjohncyh/fashion-clip` | FashionCLIP 2.0. 패션 상품 80만 건 파인튜닝, LAION CLIP ViT-B/32 기반. 사용자가 지정한 기준 후보 |
| `marqo_fashion_clip` | `Marqo/marqo-fashionCLIP` | GCL(카테고리·색상·소재까지 학습 신호로 사용). 자사 벤치마크 기준 FashionCLIP 2.0 대비 recall@1 +22% |
| `marqo_fashion_siglip` | `Marqo/marqo-fashionSigLIP` | 위와 같은 레시피의 SigLIP 판, +57%. **현재 공개된 패션 임베딩 중 보고 성능이 가장 높다** |
| `zooclaw_fashion_siglip2` | `srpone/zooclaw-fashionsiglip2` | SigLIP2-base-384 기반 최신(2026-06) 후보. 검색 목적 증류라 기대치는 낮게 — "더 최신이 더 나은가" 대조군 |

주의할 점 2가지:

- **Marqo 두 모델은 `trust_remote_code=True` + `open_clip_torch`가 필요하다.**
  `pip install open_clip_torch`를 먼저 하지 않으면 로드 단계에서 실패한다(스윕 셀은 실패한
  백엔드를 건너뛰고 사유를 표에 남긴다).
- 네 모델 모두 **검색/분류용 임베딩 모델이지 색상 전문 모델이 아니다.** 색상(§6-1)에서
  개선이 안 나오는 것 자체도 유효한 결론이다 — "체크포인트 교체로는 색상이 안 풀린다"는
  근거가 되면 VLM 유지 결정을 굳힐 수 있다.

## 사용법

1. §2(대분류)·§3(대분류 VLM)·§5(색상 VLM 계열)는 `RUN_VLM = True`로 바꿔야 실행된다
   (Qwen3-VL 로드 — 가볍지 않다).
2. §2-2·§6-1(패션 체크포인트 스윕)은 `RUN_FASHION_SWEEP = True`로 바꿔야 실행된다.
   최초 실행 시 체크포인트 4종을 내려받는다(합계 약 2GB).
3. 정답셋은 `product_info_check.ipynb`와 같은 `data/ground_truth.json`을 그대로 쓴다 —
   대분류 정답은 `taxonomy.major_of(subcategory)`로 즉석에서 계산한다(별도 라벨링 불필요).

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

import pandas as pd

from classify import classify_major_only, get_category_classifier
from color import PALETTE, build_swatch_reference
from dataset import load_ground_truth
from taxonomy import MAJOR_CATEGORIES, major_of

gt = load_ground_truth()
real_samples = [s for s in gt.samples if not s.synthetic]
print(f"정답셋: {len(gt.samples)}행(문제 {len(gt.issues)}행), 실제 사진 {len(real_samples)}장")
print(f"대분류 목록: {MAJOR_CATEGORIES}")


정답셋: 18행(문제 0행), 실제 사진 17장
대분류 목록: ('OUTER', 'TOP', 'BOTTOM', 'SHOES', 'BAG', 'ACCESSORY')


## 1. 설정 상수

`RUN_VLM`을 `True`로 바꿔야 아래 Qwen3-VL 관련 셀이 실행된다. `CATEGORY_TEMPLATE_KEY`는
CLIP 대분류 분류에 쓰는 템플릿(`taxonomy.MAJOR_PROMPT_TEMPLATES`의 키) — 소분류 검증에서
`en_clip_style`이 가장 좋았으므로 기본값으로 그대로 가져온다.

`RUN_FASHION_SWEEP`은 §2-2·§6-1(패션 특화 체크포인트 스윕)의 스위치다. 스윕은
`CATEGORY_BACKEND`(범용 baseline)와 `FASHION_SWEEP_BACKENDS`를 **같은 템플릿·같은 정답셋**
으로 돌려 비교한다 — 비교 축을 체크포인트 하나로 고정하려고 프롬프트는 건드리지 않는다.

In [ ]:
from classify import FASHION_CATEGORY_BACKENDS

RUN_VLM = True  # <- 명시적으로 True로 바꿔야 Qwen3-VL 관련 셀이 실행됨
RUN_FASHION_SWEEP = True  # <- 패션 특화 체크포인트 스윕(§2-2, §6-1). 최초 실행 시 약 2GB 다운로드
CATEGORY_BACKEND = "clip_vit_b32"  # classify.CATEGORY_BACKENDS의 키(범용 baseline)
CATEGORY_TEMPLATE_KEY = "en_clip_style"  # taxonomy.MAJOR_PROMPT_TEMPLATES의 키
FASHION_SWEEP_BACKENDS = list(FASHION_CATEGORY_BACKENDS)

print(f"CATEGORY_BACKEND={CATEGORY_BACKEND}, CATEGORY_TEMPLATE_KEY={CATEGORY_TEMPLATE_KEY}")
print(f"RUN_VLM={RUN_VLM}, RUN_FASHION_SWEEP={RUN_FASHION_SWEEP}")
print(f"FASHION_SWEEP_BACKENDS={FASHION_SWEEP_BACKENDS}")

## 2. 대분류 분류 — CLIP zero-shot

`classify.classify_major_only()` — 대분류 6종만 물어본다(소분류 30-way보다 후보가 훨씬
적어 CLIP이 더 잘할 가능성이 있다).

In [3]:
clip_classifier = get_category_classifier(CATEGORY_BACKEND)

clip_major_rows = []
for s in real_samples:
    result = classify_major_only(clip_classifier, s.image_path, CATEGORY_TEMPLATE_KEY)
    truth_major = major_of(s.subcategory)
    clip_major_rows.append(
        {
            "image_file": s.image_file,
            "truth_major": truth_major,
            "pred_major": result.predicted,
            "correct": result.predicted == truth_major,
            "score": round(result.score, 3),
        }
    )

clip_major_df = pd.DataFrame(clip_major_rows)
clip_major_acc = clip_major_df["correct"].mean()
print(f"[CLIP {CATEGORY_BACKEND}/{CATEGORY_TEMPLATE_KEY}] 대분류 top1 = {clip_major_acc:.2f}")
clip_major_df


c:\Users\SSAFY\miniforge3\envs\qual\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 398/398 [00:00<00:00, 21287.90it/s]


[CLIP clip_vit_b32/en_clip_style] 대분류 top1 = 0.35


,image_file,truth_major,pred_major,correct,score
0,denim.jpg,BOTTOM,BOTTOM,True,0.592
1,black_washed.jpg,TOP,OUTER,False,0.909
2,fix.jpg,BOTTOM,BOTTOM,True,0.518
3,levis_patch.jpg,BOTTOM,OUTER,False,0.413
4,logo_test.jpg,TOP,OUTER,False,0.825
5,scratch.jpg,BOTTOM,OUTER,False,0.490
6,small_pro.jpg,TOP,OUTER,False,0.680
7,navy_logo.png,TOP,OUTER,False,0.923
8,vintage_t.png,TOP,OUTER,False,0.712
9,black_stain.png,TOP,OUTER,False,0.730


### 2-1. 대분류 분류 — CLIP은 소분류(30-way)로 물은 뒤 대분류로 역산

방금 §2에서 CLIP에게 대분류를 직접 물으면 "OUTER" 쏠림으로 0.20까지 떨어지는 걸 봤다.
반면 소분류 30-way(`en_clip_style`)는 `product_info_check.ipynb`/`REPORT.md` §8에서
0.71~0.90이 나왔었다 — **CLIP에게는 소분류로 물어서 받고, 사용자에게는
`taxonomy.major_of()`로 역산한 대분류만 보여주면 정확도를 그대로 가져갈 수 있지
않을까**(2026-08-03, 사용자 제안)를 실측으로 확인한다.

In [5]:
from classify import classify_one_stage

clip_sub_to_major_rows = []
for s in real_samples:
    result = classify_one_stage(clip_classifier, s.image_path, CATEGORY_TEMPLATE_KEY)
    truth_major = major_of(s.subcategory)
    pred_major = major_of(result.predicted)
    clip_sub_to_major_rows.append(
        {
            "image_file": s.image_file,
            "truth_subcategory": s.subcategory,
            "pred_subcategory": result.predicted,
            "truth_major": truth_major,
            "pred_major": pred_major,
            "correct_subcategory": result.predicted == s.subcategory,
            "correct_major": pred_major == truth_major,
        }
    )

clip_sub_to_major_df = pd.DataFrame(clip_sub_to_major_rows)
clip_sub_to_major_acc = clip_sub_to_major_df["correct_major"].mean()
print(
    f"[CLIP {CATEGORY_BACKEND}/{CATEGORY_TEMPLATE_KEY}, 소분류->대분류 역산] "
    f"대분류 top1 = {clip_sub_to_major_acc:.2f}  (직접 대분류 질의 대비 {clip_major_acc:.2f})"
)
clip_sub_to_major_df


[CLIP clip_vit_b32/en_clip_style, 소분류->대분류 역산] 대분류 top1 = 1.00  (직접 대분류 질의 대비 0.35)


,image_file,truth_subcategory,pred_subcategory,truth_major,pred_major,correct_subcategory,correct_major
0,denim.jpg,DENIM,DENIM,BOTTOM,BOTTOM,True,True
1,black_washed.jpg,T_SHIRT,T_SHIRT,TOP,TOP,True,True
2,fix.jpg,DENIM,DENIM,BOTTOM,BOTTOM,True,True
3,levis_patch.jpg,DENIM,DENIM,BOTTOM,BOTTOM,True,True
4,logo_test.jpg,T_SHIRT,SHIRT,TOP,TOP,False,True
5,scratch.jpg,DENIM,DENIM,BOTTOM,BOTTOM,True,True
6,small_pro.jpg,T_SHIRT,SHIRT,TOP,TOP,False,True
7,navy_logo.png,T_SHIRT,T_SHIRT,TOP,TOP,True,True
8,vintage_t.png,T_SHIRT,T_SHIRT,TOP,TOP,True,True
9,black_stain.png,T_SHIRT,SHIRT,TOP,TOP,False,True


### 2-2. 대분류 분류 — 패션 도메인 특화 체크포인트 스윕

§2/§2-1은 **범용** 체크포인트(OpenAI CLIP ViT-B/32) 하나만 봤다. 여기서는 체크포인트만
패션 특화로 갈아끼우고 나머지(정답셋·템플릿·스코어링)는 전부 고정한 채 같은 두 방식을
다시 잰다:

- **직접 대분류 질의**(§2 방식) — 여기가 0.35로 무너진 지점이다. 패션 파인튜닝이
  "아우터 vs 상의"처럼 도메인 구분을 실제로 배웠다면 여기서 올라야 한다.
- **소분류 질의 후 역산**(§2-1 방식) — baseline이 이미 1.00이라 **올라갈 여지가 없다.**
  여기서 볼 건 개선이 아니라 **회귀 여부**(N=17이라 1.00은 쉽게 깨진다)와, 부수적으로
  같이 재는 **소분류 30-way 정확도**다. 최종 서비스가 소분류까지 노출한다면 대분류가
  아니라 이 숫자가 실제 선택 기준이 된다.

로드에 실패한 백엔드(예: `open_clip_torch` 미설치)는 스윕을 중단시키지 않고 사유만 표에
남긴다 — 4종 중 일부만 돌아도 비교는 성립한다.

In [ ]:
def measure_category_backend(backend_key, template_key):
    """백엔드 하나로 대분류 2방식 + 소분류를 한 번에 잰다.

    이미지당 질의 2회(6-way, 30-way)를 돌면서 세 지표를 같이 모은다 — 방식마다 따로
    루프를 돌면 모델을 다시 로드하게 돼서 스윕이 그만큼 느려진다.
    """
    classifier = get_category_classifier(backend_key)
    rows = []
    for s in real_samples:
        truth_major = major_of(s.subcategory)
        direct = classify_major_only(classifier, s.image_path, template_key)
        one = classify_one_stage(classifier, s.image_path, template_key)
        inverted_major = major_of(one.predicted)
        rows.append(
            {
                "image_file": s.image_file,
                "truth_major": truth_major,
                "direct_major": direct.predicted,
                "direct_correct": direct.predicted == truth_major,
                "truth_subcategory": s.subcategory,
                "pred_subcategory": one.predicted,
                "subcategory_correct": one.predicted == s.subcategory,
                "inverted_major": inverted_major,
                "inverted_correct": inverted_major == truth_major,
            }
        )
    detail = pd.DataFrame(rows)
    return detail, {
        "backend": backend_key,
        "direct_major_acc": detail["direct_correct"].mean(),
        "inverted_major_acc": detail["inverted_correct"].mean(),
        "subcategory_acc": detail["subcategory_correct"].mean(),
        "error": "",
    }


fashion_detail = {}
fashion_sweep_rows = []
if RUN_FASHION_SWEEP:
    for backend_key in [CATEGORY_BACKEND, *FASHION_SWEEP_BACKENDS]:
        try:
            detail, summary = measure_category_backend(backend_key, CATEGORY_TEMPLATE_KEY)
        except Exception as exc:  # noqa: BLE001 — 한 백엔드 실패로 스윕 전체를 죽이지 않는다
            print(f"[SKIP] {backend_key}: {type(exc).__name__}: {exc}")
            fashion_sweep_rows.append(
                {
                    "backend": backend_key,
                    "direct_major_acc": float("nan"),
                    "inverted_major_acc": float("nan"),
                    "subcategory_acc": float("nan"),
                    "error": f"{type(exc).__name__}: {exc}"[:120],
                }
            )
            continue
        fashion_detail[backend_key] = detail
        fashion_sweep_rows.append(summary)
        print(
            f"[{backend_key}] 직접 대분류 {summary['direct_major_acc']:.2f} / "
            f"역산 대분류 {summary['inverted_major_acc']:.2f} / "
            f"소분류 30-way {summary['subcategory_acc']:.2f}"
        )

    fashion_sweep_df = pd.DataFrame(fashion_sweep_rows)
    display(fashion_sweep_df)
else:
    fashion_sweep_df = pd.DataFrame(columns=["backend", "direct_major_acc", "error"])
    print("RUN_FASHION_SWEEP=False — §1 셀에서 True로 바꾸고 재실행할 것.")

In [ ]:
# 스윕에서 직접 대분류 질의가 가장 좋았던 백엔드의 오답만 본다 — 스코어 하나로는
# "무엇으로 잘못 쏠리는지"를 못 본다. §2 baseline은 전부 OUTER로 쏠렸는데(17장 중 12장),
# 패션 특화 모델도 같은 쏠림을 반복하는지 아니면 다른 방식으로 틀리는지가 다음 수를 가른다.
if RUN_FASHION_SWEEP and fashion_detail:
    best_key = max(fashion_detail, key=lambda k: fashion_detail[k]["direct_correct"].mean())
    best_detail = fashion_detail[best_key]
    print(f"직접 대분류 질의 최고 백엔드: {best_key}")
    print("직접 질의 예측 분포:")
    print(best_detail["direct_major"].value_counts().to_string())
    print(f"\n오답 {int((~best_detail['direct_correct']).sum())}건:")
    display(
        best_detail.loc[
            ~best_detail["direct_correct"],
            ["image_file", "truth_major", "direct_major", "pred_subcategory"],
        ]
    )
else:
    print("RUN_FASHION_SWEEP=False 이거나 성공한 백엔드가 없다 — §2-2를 먼저 실행할 것.")

## 3. 대분류 분류 — VLM(Qwen3-VL)

`vlm_control.query_major_category()` — 대분류 6종 프롬프트에 OUTER/TOP, BAG/ACCESSORY
구분 기준을 처음부터 넣어뒀다(소분류 v4의 교훈: 구조적/관찰 가능한 기준만 사용).

In [6]:
if RUN_VLM:
    from vlm_control import query_major_category

    vlm_major_rows = []
    for s in real_samples:
        pred_major, raw = query_major_category(s.image_path)
        truth_major = major_of(s.subcategory)
        vlm_major_rows.append(
            {
                "image_file": s.image_file,
                "truth_major": truth_major,
                "pred_major": pred_major,
                "correct": pred_major == truth_major,
                "raw": raw,
            }
        )
    vlm_major_df = pd.DataFrame(vlm_major_rows)
    vlm_major_acc = vlm_major_df["correct"].mean()
    print(f"[VLM] 대분류 top1 = {vlm_major_acc:.2f}  (CLIP 대비 {clip_major_acc:.2f})")
    display(vlm_major_df)
else:
    print("RUN_VLM=False — 위 §1 셀에서 True로 바꾸고 재실행할 것.")


W0803 13:02:03.138000 14484 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Loading weights: 100%|██████████| 713/713 [00:05<00:00, 128.67it/s]


[VLM] 대분류 top1 = 0.94  (CLIP 대비 0.35)


,image_file,truth_major,pred_major,correct,raw
0,denim.jpg,BOTTOM,BOTTOM,True,BOTTOM
1,black_washed.jpg,TOP,TOP,True,TOP
2,fix.jpg,BOTTOM,BOTTOM,True,BOTTOM
3,levis_patch.jpg,BOTTOM,BOTTOM,True,BOTTOM
4,logo_test.jpg,TOP,TOP,True,TOP
5,scratch.jpg,BOTTOM,BOTTOM,True,BOTTOM
6,small_pro.jpg,TOP,TOP,True,TOP
7,navy_logo.png,TOP,TOP,True,TOP
8,vintage_t.png,TOP,TOP,True,TOP
9,black_stain.png,TOP,TOP,True,TOP


## 4. 대분류 비교 요약

지금까지의 대분류 실측을 한 표에 모아 정확도 내림차순으로 정렬한다 — 범용 CLIP(§2/§2-1),
패션 특화 체크포인트 4종(§2-2), VLM(§3)이 같은 정답셋 위에서 어디에 서는지 본다.

읽을 때 주의: **"직접 대분류 질의"와 "소분류 질의 후 역산"은 비용이 다르지 않다**(둘 다
이미지 인코딩 1회 + 텍스트 라벨 6개 또는 30개). 즉 역산 쪽이 더 정확하다면 직접 질의를
쓸 이유가 없다. 반대로 패션 특화 모델이 직접 질의를 크게 끌어올렸다면, 그건 정확도보다
**소분류 taxonomy 변경에 덜 흔들린다**는 운영상 이점이 근거가 된다.

In [ ]:
summary_rows = [
    {
        "method": f"CLIP 직접 대분류 질의({CATEGORY_BACKEND}/{CATEGORY_TEMPLATE_KEY})",
        "major_acc": clip_major_acc,
    },
    {
        "method": f"CLIP 소분류 질의 후 역산({CATEGORY_BACKEND}/{CATEGORY_TEMPLATE_KEY})",
        "major_acc": clip_sub_to_major_acc,
    },
]
if RUN_FASHION_SWEEP:
    # §2-2 스윕 결과를 같은 표에 편다 — baseline(CATEGORY_BACKEND)은 위 두 행과 중복이라
    # 뺀다(같은 설정을 두 번 잰 것이므로 값도 같아야 한다. 다르면 재현성 문제다).
    for row in fashion_sweep_rows:
        if row["backend"] == CATEGORY_BACKEND or row["error"]:
            continue
        summary_rows.append(
            {"method": f"{row['backend']} 직접 대분류 질의", "major_acc": row["direct_major_acc"]}
        )
        summary_rows.append(
            {
                "method": f"{row['backend']} 소분류 질의 후 역산",
                "major_acc": row["inverted_major_acc"],
            }
        )
if RUN_VLM:
    summary_rows.append(
        {"method": "VLM(qwen3-vl-4b-instruct) 직접 대분류 질의", "major_acc": vlm_major_acc}
    )
pd.DataFrame(summary_rows).sort_values("major_acc", ascending=False, ignore_index=True)

## 5. 색상 분류 — 기존 설정 그대로 + 프롬프트/전처리 고도화 재실측

`product_info_check.ipynb`와 같은 정답셋·팔레트를 그대로 쓴다. 지금까지 나온 개선안을
전부 한 자리에서 비교한다:

- CLIP zero-shot(`clip_vit_b32`) — 기준선
- VLM v1(화이트밸런스 기본 적용, `query_color()` 기본값) — `REPORT.md` §13 실측 0.90
- VLM v2_anchored / v3_khaki_gray + 화이트밸런스 — **이 조합은 처음 테스트한다.**
  지난번 v2/v3 실측(§12)은 화이트밸런스 적용 **전**이었다 — 조명 편향 자체가 v2/v3
  회귀의 숨은 원인이었을 수 있어서, 화이트밸런스로 입력을 먼저 정리한 뒤에도 v2/v3가
  여전히 v1보다 나쁜지 다시 확인한다.

In [8]:
from evaluate import evaluate_color

clip_color_preds = evaluate_color(gt.samples, "clip_vit_b32")
clip_color_acc = sum(p.correct_primary for p in clip_color_preds) / len(clip_color_preds)
print(f"[CLIP clip_vit_b32] 색상 primary_acc = {clip_color_acc:.2f}")


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 19122.89it/s]


[CLIP clip_vit_b32] 색상 primary_acc = 0.41


In [9]:
if RUN_VLM:
    from vlm_control import COLOR_PROMPT_VERSIONS, query_color

    color_prompt_wb_rows = []
    for version in COLOR_PROMPT_VERSIONS:
        correct = 0
        for s in real_samples:
            pred_primary, _sec, _raw = query_color(s.image_path, prompt_version=version)
            correct += pred_primary == s.primary_color
        color_prompt_wb_rows.append(
            {
                "prompt_version": version,
                "white_balance": True,
                "n": len(real_samples),
                "primary_acc": correct / len(real_samples),
            }
        )
    color_prompt_wb_df = pd.DataFrame(color_prompt_wb_rows)
    display(color_prompt_wb_df)
else:
    print("RUN_VLM=False — §1 셀에서 True로 바꾸고 재실행할 것.")


,prompt_version,white_balance,n,primary_acc
0,v1,True,17,0.647059
1,v2_anchored,True,17,0.705882
2,v3_khaki_gray,True,17,0.705882


## 6. 색상 분류 — 팔레트 스와치 참조(색상 개선안 4번)

`vlm_control.query_color_with_swatches()` — 상품 사진 아래에 팔레트 hex 스와치 그리드를
합성해 같은 이미지로 같이 보여준다(`color.build_swatch_reference()`). 텍스트로 색 이름을
"기억"해서 매칭하는 대신, 이미지 대 이미지로 비교 판단하게 유도하는 방법 — 여태 프롬프트
문구만 바꾸는 시도(v2/v3)는 다 whack-a-mole이었으니, 아예 다른 축(시각적 참조)을 시도해
본다. 화이트밸런스는 여기서도 기본 적용된다(상품 사진 쪽에만 적용, 스와치는 고정 hex라
보정 대상이 아니다).

In [10]:
if RUN_VLM:
    from vlm_control import query_color_with_swatches

    swatch_rows = []
    for s in real_samples:
        pred_primary, pred_secondary, _raw = query_color_with_swatches(s.image_path)
        swatch_rows.append(
            {
                "image_file": s.image_file,
                "truth_primary": s.primary_color,
                "pred_primary": pred_primary,
                "pred_secondary": pred_secondary,
                "correct": pred_primary == s.primary_color,
            }
        )
    swatch_df = pd.DataFrame(swatch_rows)
    swatch_acc = swatch_df["correct"].mean()
    print(f"[VLM + 스와치 참조] 색상 primary_acc = {swatch_acc:.2f}")
    display(swatch_df)
else:
    print("RUN_VLM=False — §1 셀에서 True로 바꾸고 재실행할 것.")


[VLM + 스와치 참조] 색상 primary_acc = 0.47


,image_file,truth_primary,pred_primary,pred_secondary,correct
0,denim.jpg,BLUE,GRAY,BEIGE,False
1,black_washed.jpg,BLACK,GRAY,NAVY,False
2,fix.jpg,BLUE,BEIGE,NAVY,False
3,levis_patch.jpg,BLUE,GRAY,NAVY,False
4,logo_test.jpg,WHITE,WHITE,NaN,True
5,scratch.jpg,BLUE,BEIGE,NAVY,False
6,small_pro.jpg,WHITE,IVORY,BEIGE,False
7,navy_logo.png,NAVY,NAVY,BLUE,True
8,vintage_t.png,KHAKI,KHAKI,GRAY,True
9,black_stain.png,BLACK,NAVY,NaN,False


## 6-1. 색상 분류 — 패션 도메인 특화 체크포인트 스윕

색상 쪽에서 지금까지 시도한 축은 전부 소진됐다: 프롬프트 앙상블(§11) 회귀, VLM 프롬프트
v2/v3(§12·§15) 무의미, 스와치 참조(§16) 회귀. 유효했던 건 화이트밸런스 + content-free
편향 보정뿐이고(§17, 0.20→0.50) 그래도 VLM(0.90)에 못 미친다.

남은 축은 **체크포인트 교체**다. §2-2와 같은 패션 특화 모델 4종을 색상 라벨에도 그대로
물어본다 — 특히 Marqo 두 모델은 색상을 학습 신호로 명시적으로 썼다고 밝히고 있어, 여기서
안 오르면 "CLIP 계열 zero-shot으로 색상은 안 풀린다"는 결론의 근거가 꽤 강해진다.

화이트밸런스는 §13/§17에서 유일하게 재현된 개선이므로 각 모델을 **원본 / `_wb`** 두 조건으로
같이 잰다 — 전처리 효과가 체크포인트에 따라 달라지는지도 함께 보려는 것이다.

In [ ]:
# 비교 대상은 §5의 CLIP 색상 baseline과 같은 정답셋(gt.samples)이다 — VLM 셀들이 쓰는
# real_samples(N=17)와 행 수가 다르므로 CLIP끼리만 직접 비교한다.
fashion_color_rows = []
if RUN_FASHION_SWEEP:
    color_backend_keys = [CATEGORY_BACKEND, f"{CATEGORY_BACKEND}_wb"]
    for key in FASHION_SWEEP_BACKENDS:
        color_backend_keys += [key, f"{key}_wb"]

    for key in color_backend_keys:
        try:
            preds = evaluate_color(gt.samples, key)
        except Exception as exc:  # noqa: BLE001 — 백엔드 하나 실패로 스윕을 멈추지 않는다
            print(f"[SKIP] {key}: {type(exc).__name__}: {exc}")
            fashion_color_rows.append(
                {
                    "backend": key,
                    "n": 0,
                    "primary_acc": float("nan"),
                    "error": f"{type(exc).__name__}: {exc}"[:120],
                }
            )
            continue
        acc = sum(p.correct_primary for p in preds) / len(preds)
        fashion_color_rows.append(
            {"backend": key, "n": len(preds), "primary_acc": acc, "error": ""}
        )
        print(f"[{key}] 색상 primary_acc = {acc:.2f}")

    fashion_color_df = pd.DataFrame(fashion_color_rows).sort_values(
        "primary_acc", ascending=False, ignore_index=True
    )
    display(fashion_color_df)
else:
    fashion_color_df = pd.DataFrame(columns=["backend", "n", "primary_acc", "error"])
    print("RUN_FASHION_SWEEP=False — §1 셀에서 True로 바꾸고 재실행할 것.")

## 7. 색상 비교 요약

지금까지 실측한 색상 방법론을 전부 한 표에 모아 정확도 내림차순으로 정렬한다 —
`REPORT.md` §8~§17에 흩어져 있던 CLIP/VLM/프롬프트/전처리/스와치 결과에 §6-1의
패션 특화 체크포인트까지 더한 최종 스코어보드.

CLIP 행과 VLM 행은 정답셋 행 수가 다르다(CLIP은 `gt.samples`, VLM은 `real_samples`) —
CLIP끼리·VLM끼리의 비교는 정확하지만 두 계열 사이의 소수점 차이는 그대로 읽지 말 것.

In [ ]:
color_summary_rows = [{"method": "CLIP zero-shot(clip_vit_b32)", "primary_acc": clip_color_acc}]
if RUN_FASHION_SWEEP:
    for row in fashion_color_rows:
        if row["backend"] == CATEGORY_BACKEND or row["error"]:
            continue  # baseline은 위 행과 중복, 실패한 백엔드는 숫자가 없다
        color_summary_rows.append(
            {"method": f"CLIP zero-shot({row['backend']})", "primary_acc": row["primary_acc"]}
        )
if RUN_VLM:
    for _, row in color_prompt_wb_df.iterrows():
        color_summary_rows.append(
            {"method": f"VLM {row['prompt_version']} + 화이트밸런스", "primary_acc": row["primary_acc"]}
        )
    color_summary_rows.append({"method": "VLM v1 + 스와치 참조", "primary_acc": swatch_acc})
pd.DataFrame(color_summary_rows).sort_values("primary_acc", ascending=False, ignore_index=True)

## 참고

- 소분류(30종) 결과·카테고리 프롬프트 v1~v4·색상 §8~§13 전체 서술은
  `product_info_check.ipynb`와 `REPORT.md` 참고.
- 이 노트북에서 나온 의미 있는 결과는 `REPORT.md`에도 반영해 둘 것 — 출력은 재실행하면
  사라진다.